# colabutilscs4337 release smoke test

Run this **before approving a PyPI release** (Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all).
It installs the release candidate from TestPyPI and exercises everything that CI cannot: the GPU,
Google Drive, notebook-name detection, Netron in the cell output, and a real fastai training run.

Every check asserts; if the last cell prints `ALL SMOKE TESTS PASSED`, the release is good.

In [ ]:
VERSION = ""          # release candidate, e.g. "0.1.0"; empty = newest
SOURCE = "testpypi"   # "testpypi" before release, "pypi" to re-check a published version

import subprocess, sys
before = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                        capture_output=True, text=True).stdout
spec = f"colabutilscs4337=={VERSION}" if VERSION else "colabutilscs4337"
if SOURCE == "testpypi":
    %pip install -q --upgrade --no-deps --index-url https://test.pypi.org/simple/ {spec}
else:
    %pip install -q --upgrade {spec}

In [ ]:
import colabutilscs4337 as cu

print("colabutilscs4337", cu.__version__)
assert not VERSION or cu.__version__ == VERSION, cu.__version__
assert cu.env.is_colab(), "Run this notebook in Google Colab"
info = cu.env.runtime_info()
print(info["python"], info["gpus"])
assert info["gpus"], "Select a GPU runtime (Runtime > Change runtime type)"
rows = cu.env.check()

## Locked install of the optional extras changes nothing that Colab preinstalled

In [ ]:
result = cu.install("all")
after = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                       capture_output=True, text=True).stdout
parse = lambda text: dict(l.split("==", 1) for l in text.splitlines() if "==" in l)
b, a = parse(before), parse(after)
changed = {k: (b[k], v) for k, v in a.items() if k in b and b[k] != v and k != "colabutilscs4337"}
print("added:", sorted(set(a) - set(b)))
assert not changed, f"Preinstalled packages changed: {changed}"
assert not result.new_problems, result.new_problems

## Notebook name and Google Drive run folder

In [ ]:
name = cu.env.notebook_name()
print("notebook:", name)
assert name != "untitled", "notebook_name() fell back to the default; check _colab.py sources"

run = cu.drive.new_run("smoke-test")
print(run)
assert str(run).startswith("/content/drive/MyDrive/runs/")
assert (run / "run.json").is_file()
(run / "hello.txt").write_text("ok")
assert cu.drive.latest_run(name) == run.path
print(cu.drive.get_config())

## PyTorch model: summary, Netron, save and load

In [ ]:
import torch, torchvision

device = "cuda"
model = torchvision.models.resnet18(weights="DEFAULT").to(device).eval()
stats = cu.tools.summary(model, input_size=(1, 3, 224, 224), verbose=0)
assert stats.total_params == 11_689_512, stats.total_params
print(stats)

cu.tools.netron(model, height=500)   # an interactive graph should appear below

In [ ]:
torch.save(model.state_dict(), run / "resnet18.pth")
loaded = cu.models.load_model(run / "resnet18.pth", arch=torchvision.models.resnet18)
x = torch.randn(2, 3, 224, 224, device=device)
assert next(loaded.parameters()).is_cuda
assert torch.allclose(loaded(x), model(x), atol=1e-4)
cu.tools.stop_netron()

## fastai: train briefly, export to Drive, load back

In [ ]:
from fastai.vision.all import *

path = untar_data(URLs.MNIST_TINY)
dls = ImageDataLoaders.from_folder(path, train="train", valid="valid", item_tfms=Resize(64), bs=32)
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.fine_tune(1)
learn.path = run.path
learn.export("export.pkl")
learn.save("ckpt")

m1 = cu.models.load_model(run / "export.pkl")
m2 = cu.models.load_model(run / "models" / "ckpt.pth", arch=learn)
xb = dls.one_batch()[0]
with torch.no_grad():
    ref = learn.model.eval()(xb)
    assert torch.allclose(m1(xb.to(next(m1.parameters()).device)).cpu(), ref.cpu(), atol=1e-4)
    assert torch.allclose(m2(xb).cpu(), ref.cpu(), atol=1e-4)

## Grad-CAM and activations

In [ ]:
img = get_image_files(path / "valid")[0]
xb = cu.viz.learner_input(learn, img)
layer = cu.viz.suggest_target_layer(learn, xb)
print("target layer:", layer)

cam = cu.viz.gradcam(learn, xb)
assert cam.cams.shape[-2:] == xb.shape[-2:]
cu.viz.show_cam(xb, cam, mean=cu.viz.IMAGENET_MEAN, std=cu.viz.IMAGENET_STD)

cam_pp = cu.viz.gradcam(learn, xb, method="gradcam++")
cu.viz.show_cam(xb, cam_pp, mean=cu.viz.IMAGENET_MEAN, std=cu.viz.IMAGENET_STD)

acts = cu.viz.activations(learn, xb, layers=["0.4"])
cu.viz.show_feature_maps(acts["0.4"], n=16, title="body block 4")

In [ ]:
print("ALL SMOKE TESTS PASSED for colabutilscs4337", cu.__version__)